# 浩劫原生自对弈训练
在 Colab 选择 **L4 / A100 或其他支持原生 BF16 的 GPU**。T4、P100 不支持本训练链路。

依次点击各单元左侧的运行按钮：安装最新版 → 选择检查点 → 点击“启动训练”。也可先点“全部运行”，最后点击“启动训练”。全程无需编辑代码，安装默认使用 `main`。

源码与依赖放在 `/content`；运行目录和定时检查点放在 Google Drive。默认从头开始完整终局训练；后续可下拉选择本版本的检查点续训。旧短轨迹检查点保留，但不用于新训练。保存到新目录，保存间隔默认 300 秒，也可手动保存或停止。

启动后自动显示 Web 控制台：吞吐、损失、策略滞后、资源曲线、保存/停止/并发控制，以及手工和定期 AI 测评。默认每 30 分钟测评一轮（12 局），可修改或关闭。重跑最后一个单元可重开面板；关闭面板不停止训练或定期测评。更新前先点“保存并停止”，等退出后重新运行安装、选择和启动单元，无需重启运行时。

In [ ]:
#@title 1. 安装或更新（main 最新版）
from pathlib import Path
import json
import subprocess
from datetime import datetime, timezone

existing_process = globals().get('process')
if existing_process is not None and existing_process.poll() is None:
    raise RuntimeError('训练仍在运行。请先保存并停止，等待进程退出后再更新安装；不用重启 Colab 运行时。')

REPOSITORY = 'https://github.com/Loremmlel/haojie2.git'
REVISION = 'main'
SOURCE = Path('/content/haojie2')
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], check=True)
subprocess.run(['bash', str(SOURCE / 'training/colab/setup.sh')], check=True)
NATIVE = ['bash', str(SOURCE / 'training/native/run.sh')]

In [ ]:
#@title 2. 选择训练起点（默认从头开始）
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display

existing_process = globals().get('process')
if existing_process is not None and existing_process.poll() is None:
    raise RuntimeError('训练仍在运行，请在控制台保存并停止后再选择。')
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/haojie-training')
checkpoints = sorted(ROOT.glob('*/checkpoint.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
checkpoint_picker = widgets.Dropdown(
    options=[('从头开始完整终局训练（保留旧档）', '')] + [(f'继续：{p.parent.name}', str(p)) for p in checkpoints],
    description='训练起点', layout=widgets.Layout(width='100%'))
configuration_output = widgets.Output()

def prepare_training(change=None):
    global ARGS, configuration
    configuration = None
    ARGS = ['--device', 'cuda']
    if checkpoint_picker.value:
        ARGS += ['--resume', checkpoint_picker.value]
    with configuration_output:
        configuration_output.clear_output(wait=True)
        try:
            configuration = json.loads(subprocess.check_output([*NATIVE, 'configure', *ARGS], text=True, stderr=subprocess.STDOUT))
            print('恢复检查点：', checkpoint_picker.value or '无，从头开始')
            print(json.dumps(configuration, ensure_ascii=False, indent=2))
        except subprocess.CalledProcessError as error:
            print('配置失败，请检查 GPU 或选择其他检查点：\n', error.output)

checkpoint_picker.observe(prepare_training, names='value')
display(checkpoint_picker, configuration_output)
prepare_training()

In [ ]:
#@title 3. 启动训练与 Web 控制台
import sys
import importlib
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
from training.colab import panel

def show_console():
    global web_panel
    web_panel = panel.show(RUN, process, configuration, native=NATIVE,
                           python='/content/haojie-dependencies/venv/bin/python')

existing_process = globals().get('process')
if existing_process is not None and existing_process.poll() is None:
    show_console()
elif checkpoint_picker.disabled:
    print('训练已结束。请重新运行第 2 单元选择最新检查点，再运行本单元。')
else:
    if panel._current and panel._current.evaluation:
        panel._current.evaluation.close()
    importlib.reload(panel)
    start_button = widgets.Button(description='启动训练', button_style='primary')
    start_output = widgets.Output()

    def start_training(button):
        global RUN, LOG, process
        existing_process = globals().get('process')
        if existing_process is not None and existing_process.poll() is None:
            return
        with start_output:
            start_output.clear_output(wait=True)
            if configuration is None:
                print('请先在上方完成配置，修正配置错误后再启动。')
                return
            button.disabled = checkpoint_picker.disabled = True
            try:
                RUN = ROOT / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
                ROOT.mkdir(parents=True, exist_ok=True)
                LOG = RUN.with_suffix('.log')
                with LOG.open('x') as output:
                    process = subprocess.Popen([*NATIVE, 'run', '--directory', str(RUN), *ARGS],
                                               stdout=output, stderr=subprocess.STDOUT)
            except OSError as error:
                button.disabled = checkpoint_picker.disabled = False
                print('启动失败，请重试：', error)
                return
            print('已启动，PID:', process.pid, '\n新运行目录:', RUN, '\n日志:', LOG)
            show_console()

    start_button.on_click(start_training)
    display(start_button, start_output)